# Geometry 04b — IMO-AG-30 : la revanche de la méthode de Wu

**Public : Recherche** — accrétion du [Geometry-04](Geometry-04-DD-AR-Python.ipynb) dans la série [Geometry](README.md). Elle suppose le noyau Wu du [Geometry-03](Geometry-03-Wu-Method-Python.ipynb) (pseudo-reste, ensemble caractéristique) et le moteur DD+AR du 04. Le chemin principal de la série se lit sans ouvrir cette page : les résultats de recherche vivent dans les accrétions `b`.

***

En 2024, **AlphaGeometry** (Trinh et al.) marque une rupture : un modèle neuro-symbolique entraîné sur 100 millions de preuves synthétiques résout **25 problèmes sur 30** du benchmark IMO-AG-30 — et l'article fondateur rapporte, en passant, qu'une « baseline » basée sur la méthode de Wu n'en résoudrait que dix. Sinha et al. (2024) repostulent cette baseline et la trouve **drastiquement sous-évaluée** : Wu seul en résout **15**, l'union Wu+DD+AR atteint **21/30 — niveau médaille d'argent, sur un laptop CPU avec 5 minutes par problème** — et l'union Wu+AlphaGeometry porte l'état de l'art à **27/30, au-dessus du niveau médaille d'or**. Deux problèmes du benchmark sont résolus **par Wu seul**, donc par aucun des systèmes neuro-symboliques évalués : IMO 2021 P3 et **IMO 2008 P1B**.

Ce carnet prend le second — IMO 2008 P1 — et le traite **avec les moteurs que la série a construits** : encodage algébrique du vrai énoncé olympique, preuve complète par le noyau Wu du 03 (relué ici en trois cellules), confrontation au littéral exact soumis aux évaluateurs (le prédicat `CYCLIC` à quatre points décodé des fichiers du benchmark), mesure de la frontière où DD s'arrête, témoin négatif, et — à échelle de jouet déclarée — la brique qui manque à DD+AR pour atteindre AlphaGeometry : le **proposeur neuronal de constructions auxiliaires**.

## Objectifs pédagogiques

1. **Traduire** un énoncé d'olympiade réel (orthocentre, trois cercles, six points, concyclicité) en système polynomial — le geste des carnets 01 et 03, sur un sujet qui a résisté à un système entraîné sur 100 millions d'exemples.
2. **Exécuter** le noyau Wu de la série sur ce système et prouver l'énoncé officiel **en entier** (les six points), pas seulement sa découpe d'évaluation.
3. **Lire** les non-dégénérescences produites par la preuve et leur sens géométrique.
4. **Comparer** la preuve au littéral du benchmark (déterminant de cyclicité à quatre points) et la certifier par un second moteur indépendant (Gröbner, astuce de Rabinowitsch).
5. **Mesurer** où DD s'arrête sur cette configuration, ce que Wu garantit que DD ne peut pas dire — et ce qu'un proposeur neuronal apporte (et n'apporte pas), à échelle de jouet.

**Références** (primaire d'abord) : Sinha, Prabhu, Kumaraguru, Bhat, Bethge, *Wu's Method can Boost Symbolic AI to Rival Silver Medalists and AlphaGeometry to Outperform Gold Medalists at IMO Geometry*, 2024 — PDF archivé dans le gisement commun (`G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\`) ; les artefacts du benchmark (`translated_imo_2008_p1a.gex`, `translated_imo_2008_p1b.gex` du dépôt HuggingFace `bethgelab/simplegeometry`) décodés au §2 ; l'énoncé officiel IMO 2008 (notes Evan Chen, IMO register, wiki AoPS). Trinh et al. 2024 est cité **de seconde main**, via le résumé qu'en font Sinha et al. : son contenu détaillé n'est pas une dépendance de ce carnet.

## Plan du notebook

| § | Contenu | Livrable |
|---|---|---|
| 1 | Le benchmark IMO-AG-30 et les scores rapportés | tableau sourcé |
| 2 | Le problème : énoncé officiel 2008 P1, découpe AG-30 | les deux `SHOW` décodés |
| 3 | Encodage : repère, orthocentre, trois cercles, six contraintes | `H_IMO` triangulaire |
| 4 | La preuve par le noyau Wu du 03 | **énoncé officiel prouvé** + non-dégénérescences + Gröbner croisé |
| 5 | Le littéral P1B d'AG-30 | déterminant `CYCLIC` réduit à zéro |
| 6 | La frontière de DD | fermeture mesurée, absence structurelle |
| 7 | Témoin négatif | un énoncé faux rejeté trois fois |
| 8 | Le proposeur neuronal, à échelle de jouet | classifieur entraîné dans le carnet |
| 9 | Exercices | 3 énoncés à compléter |

*Notebook de la série Geometry — SymbolicAI. Fait partie du cycle complet du raisonnement vérifiable du dépôt CoursIA.*

## 1. Le benchmark IMO-AG-30 et les scores rapportés

L'article d'AlphaGeometry a introduit le benchmark **IMO-AG-30** : 30 problèmes de géométrie des Olympiades Internationales Mathématiques, traduits dans le langage de son moteur symbolique (points, droites, cercles ; prédicats d'alignement, de perpendicularité, de cyclicité). La moitié symbolique du système — **DD** (*deductive database*, fermeture par règles) augmentée d'**AR** (*angle, ratio and distance chasing*) — prouve ce qui est combinatoirement dérivable ; la moitié neuronale — un transformer entraîné sur 100 millions de preuves synthétiques — **propose les constructions auxiliaires** qui débloquent la chaîne de déduction.

Tous les chiffres ci-dessous sont relevés dans la copie archivée de Sinha et al. 2024 ; aucun n'est reproduit de mémoire.

| Système | Score IMO-AG-30 | Conditions rapportées |
|---|---|---|
| AlphaGeometry (neuro-symbolique) | **25/30** | transformer + 100 M preuves synthétiques d'entraînement |
| Wu, baseline rapportée par l'article AG | 10/30 | mentionnée en passant, non reproduite |
| **Wu seul, réévalué** (Sinha et al.) | **15** sur 26 traduisibles | JGEX, laptop CPU, 5 min/problème ; 14 solutions en < 5 s, sinon mémoire épuisée < 5 min ; 4 problèmes non traduisibles en JGEX |
| **Wu + DD + AR** | **21/30** — niveau argent | CPU seul, 5 min/problème ; « résout 4 problèmes de moins qu'AlphaGeometry » |
| **Wu + AlphaGeometry** | **27/30** — SOTA | première méthode automatique au-dessus d'un médaillé d'or IMO |

La phrase qui fonde ce carnet : *deux des quinze problèmes résolus par Wu (2021 P3, 2008 P1B) figurent parmi les cinq qu'AlphaGeometry ne résout pas*. Une méthode de 1977, sans apprentissage, tient des énoncés qu'un transformer de cent millions d'exemples n'atteint pas. La suite examine l'un d'eux de près.

**Ce que ce carnet n'évalue pas** : AlphaGeometry lui-même (entraînement GPU de cent millions d'échantillons — hors budget d'un carnet, et son article n'est pas archivé localement : nous ne dépendons pas de son contenu détaillé). Les scores AG ci-dessus sont *rapportés* par Sinha et al., pas rejoués ici ; ce que nous exécutons, c'est la moitié SymbolicAI — Wu, DD+AR — et un proposeur à échelle de jouet.

## 2. Le problème : IMO 2008 P1, et sa découpe AG-30

> **IMO 2008, Problème 1** (énoncé officiel). Soit H l'orthocentre d'un triangle acutangle ABC. Le cercle Γ_A, de centre le milieu de [BC] et passant par H, recoupe la droite (BC) en A₁ et A₂. On définit de même B₁, B₂ (cercle de centre le milieu de [CA] passant par H, recoupant (CA)) et C₁, C₂. Montrer que les six points A₁, A₂, B₁, B₂, C₁, C₂ sont **concycliques**.

Trois sources concordantes : notes d'Evan Chen (`web.evanchen.cc/exams/IMO-2008-notes.pdf`), rapport officiel (imo-register.org.uk), wiki AoPS. L'énoncé officiel est **unique** — il n'a pas de partie (b).

D'où vient alors le « 2008 P1B » du tableau du §1 ? De la **traduction dans le langage du benchmark**. Les fichiers JGEX du dépôt `bethgelab/simplegeometry` — les artefacts exacts soumis au solveur — portent, pour ce problème, deux entrées :

```
translated_imo_2008_p1a.gex : SHOW: CYCLIC C1 C2 B1 B2
translated_imo_2008_p1b.gex : SHOW: CYCLIC C1 C2 B1 A1
```

Hypothèses identiques des deux côtés (`TRIANGLE A B C; ORTHOCENTER H; MIDPOINT D C B; ... CIRCLE D H; INTERSECTION_LC A1 B C D H; ...`) : seule la conclusion change — deux revendications de cyclicité **à quatre points**. Le prédicat du moteur (et les règles DD de cyclicité) travaille sur des quadruplets : l'énoncé officiel « les six points sont concycliques » s'y décompose naturellement en telles revendications partielles. (Les fichiers portent aussi deux points libres `K`, `M` jamais utilisés dans le `SHOW` — des résidus d'édition de l'artefact, que nous lisons pour ce qu'ils sont.)

C'est **P1B** — `CYCLIC C1 C2 B1 A1` — que Wu résout et AlphaGeometry non. Nous ferons mieux que la revendication partielle : le §4 prouve l'énoncé officiel à six points, le §5 réduit quand même le littéral P1B exact.

In [1]:
from sympy import (symbols, Poly, prem, expand, groebner, factor, Rational, sqrt,
                   solve, nsimplify, together, Matrix, lambdify)
import time
import numpy as np
import sympy

print("sympy", sympy.__version__, "| numpy", np.__version__)

sympy 1.14.0 | numpy 2.4.2


## 3. L'encodage : trois cercles, cinq inconnues auxiliaires

Le geste est celui du carnet 01, sur un vrai sujet. Repère : **B = (0,0), C = (1,0), A = (u,v)** — deux paramètres libres (u, v). L'orthocentre H = (u, u(1−u)/v) : l'altitude issue de A est la verticale x = u, et l'altitude issue de B impose l'ordonnée (éliminer le point courant de (AC) avec (BH) ⊥ (AC)).

Chaque famille de points apporte **une inconnue et une contrainte** — jamais de racine carrée dans les équations :

- **Famille A** : Γ_A a pour centre M_a = (1/2, 0) et rayon |H−M_a|. Ses intersections avec (BC) sont (1/2 ± s, 0), où s porte la contrainte s² = (u−1/2)² + (u(1−u)/v)² — dénominateurs déblayés : **h₁ : s²v² = v²(u−1/2)² + u²(1−u)²**.
- **Famille B** : un point de (CA) s'écrit (1 + z(u−1), z·v) pour un paramètre z ; « sur Γ_B de centre M_b = ((1+u)/2, v/2), rayon |H−M_b| » est un **trinôme en z**. Les deux racines sont B₁ (paramètre p) et B₂ (paramètre q) : le même trinôme, évalué en p et en q.
- **Famille C** : un point de (AB) s'écrit (z·u, z·v) ; « sur Γ_C de centre M_c = (u/2, v/2) » est un trinôme en z, évalué en r (C₁) et t (C₂).

Cinq hypothèses, une par inconnue auxiliaire : le système est **déjà triangulaire** dans l'ordre (u, v, s, p, q, r, t) — paramètres d'abord, inconnues ensuite. C'est exactement la structure que l'ensemble caractéristique du 03 cherche à produire.

**La route de la conclusion.** Prouver « six points concycliques » exige un candidat cercle. Observation d'algébriste : M_a a pour abscisse 1/2, et A₁, A₂ lui sont symétriques — tout cercle passant par A₁ et A₂ a un centre d'abscisse 1/2. Or le **circoncentre** O de ABC a précisément abscisse 1/2 (médiatrice de [BC]) : c'est le candidat naturel. Nous prouverons que les quatre points B₁, B₂, C₁, C₂ sont à la même distance de O que A₁ — l'énoncé officiel entier, avec en prime un fait que la conclusion ne demandait pas : *le cercle des six points est centré au circoncentre*.

In [2]:
# Encodage de IMO 2008 P1.
u, v = symbols("u v", real=True)            # parametres : A=(u,v), B=(0,0), C=(1,0)
s, p, q, r, t = symbols("s p q r t")        # inconnues auxiliaires (une par contrainte)
V_IMO = [u, v, s, p, q, r, t]               # ordre d'elimination : parametres, puis dependantes

hy = u*(1 - u)/v                            # orthocentre H = (u, hy)

# Famille A : Gamma_A centree en M_a=(1/2,0), passant par H ; A1=(1/2+s,0), A2=(1/2-s,0)
h1 = expand(s**2*v**2 - (v**2*(u - Rational(1, 2))**2 + u**2*(1 - u)**2))

def trinome_B(z):
    """Point de (CA) parametre par z, sur Gamma_B (centre M_b, rayon |H-M_b|) : trinome en z."""
    ex, ey = 1 + z*(u - 1) - (1 + u)/2, z*v - v/2
    rx, ry = u - (1 + u)/2, hy - v/2
    return expand(together(ex**2 + ey**2 - (rx**2 + ry**2)).as_numer_denom()[0])

def trinome_C(z):
    """Point de (AB) parametre par z, sur Gamma_C (centre M_c, rayon |H-M_c|) : trinome en z."""
    ex, ey = z*u - u/2, z*v - v/2
    rx, ry = u - u/2, hy - v/2
    return expand(together(ex**2 + ey**2 - (rx**2 + ry**2)).as_numer_denom()[0])

fB_p, fB_q = trinome_B(p), trinome_B(q)     # les deux racines de la famille B
fC_r, fC_t = trinome_C(r), trinome_C(t)     # les deux racines de la famille C
H_IMO = [h1, fB_p, fB_q, fC_r, fC_t]

for f, nom in [(h1, "h1  (famille A, var s)"), (fB_p, "fB_p (famille B, var p)"),
               (fC_r, "fC_r (famille C, var r)")]:
    mvar = [x for x in V_IMO if Poly(f, x).degree() > 0][-1]
    print(f"{nom:24s} degre {Poly(f, mvar).degree()} en {mvar}, {len(str(f))} caracteres")
print("\n5 hypotheses, 5 inconnues auxiliaires : systeme deja triangulaire.")

h1  (famille A, var s)   degre 2 en s, 62 caracteres
fB_p (famille B, var p)  degre 2 en p, 163 caracteres
fC_r (famille C, var r)  degre 2 en r, 109 caracteres

5 hypotheses, 5 inconnues auxiliaires : systeme deja triangulaire.


In [3]:
# Sanity numerique AVANT la machinerie symbolique (regle de la serie) :
# trois triangles rationnels ACUTANGLES, les six points construits par radicaux,
# la concyclicité mesuree exactement. Un encodage faux mourrait ici.
def six_points(u_, v_):
    """Les six points exacts pour A=(u_,v_), par resolution des cinq contraintes."""
    hy_ = u_*(1 - u_)/v_
    sa = sqrt((u_ - Rational(1, 2))**2 + hy_**2)
    pts = [("A1", Rational(1, 2) + sa, 0), ("A2", Rational(1, 2) - sa, 0)]
    for nom, fam, direc, orig in [("B", trinome_B, (u_ - 1, v_), (1, 0)),
                                  ("C", trinome_C, (u_, v_), (0, 0))]:
        z = symbols("z")
        racines = solve(Poly(fam(z).subs({u: u_, v: v_}), z), z)
        assert len(racines) == 2
        for k, rk in enumerate(racines, 1):
            pts.append((f"{nom}{k}", orig[0] + rk*direc[0], rk*direc[1]))
    return pts

w_sym = (u**2 + v**2 - u)/(2*v)             # ordonnee du circoncentre O=(1/2, w)

worst = 0
for (u_, v_) in [(Rational(1, 2), 1), (Rational(3, 7), Rational(5, 7)),
                 (Rational(2, 5), Rational(13, 17))]:
    assert u_*(1 - u_) < v_**2, "triangle non acutangle en A"
    pts = six_points(u_, v_)
    w_ = w_sym.subs({u: u_, v: v_})
    d2 = [((x - Rational(1, 2))**2 + (y - w_)**2) for (_, x, y) in pts]
    ref, *autres = d2
    ecart = max(abs(a - ref) for a in autres)
    worst = max(worst, float(ecart))
    print(f"A=({u_},{v_}) : |d^2(P,O) - d^2(A1,O)| max sur les 5 autres points = {ecart}")
print(f"\nPire ecart exact sur 3 triangles : {worst}  -> encodage COHERENT avec l'enonce")

A=(1/2,1) : |d^2(P,O) - d^2(A1,O)| max sur les 5 autres points = 0
A=(3/7,5/7) : |d^2(P,O) - d^2(A1,O)| max sur les 5 autres points = 0


A=(2/5,13/17) : |d^2(P,O) - d^2(A1,O)| max sur les 5 autres points = 0

Pire ecart exact sur 3 triangles : 0  -> encodage COHERENT avec l'enonce


## 4. La preuve : le noyau Wu du 03, relu en trois cellules

La convention de la série : chaque carnet **redéclare son moteur** en cellules lisibles plutôt que de l'importer — c'est le prix de comprendre *pourquoi* ça marche. Trois briques du [Geometry-03](Geometry-03-Wu-Method-Python.ipynb) suffisent ici : le pseudo-reste `prem_s` (confronté au moteur de référence `sympy.prem`), la chaîne triangulaire (`basic_set` de Chou, `char_set_basic`), et le test (`wu_prem` : pseudo-reste de la conclusion à travers la chaîne, du sommet vers la base). Rien de nouveau — l'exercice est de les faire travailler sur un énoncé qui a résisté à AlphaGeometry.

In [4]:
def prem_s(f, g, v):
    '''Pseudo-reste de f par g en la variable v (implementation pedagogique du 03).

    Retourne r tel que lc^(m-d+1) f = q g + r avec deg_v(r) < deg_v(g).
    '''
    f, g = expand(f), expand(g)
    dg = Poly(g, v).degree()
    if dg == 0:
        return f
    lc = Poly(g, v).LC()
    r = f
    while r != 0 and Poly(r, v).degree() >= dg:
        dr = Poly(r, v).degree()
        cr = Poly(r, v).LC()
        r = expand(lc * r - cr * v**(dr - dg) * g)
    return r

# Cross-check systematique contre sympy.prem (le moteur de reference)
_x, _y = symbols("x y")
cas = [(_x**3 - 2*_x*_y + 5, _x**2 - _y, _x),
       (_y*_x**2 + 3*_x + 1, _x - _y, _x),
       (_x**2*_y**2 + _x*_y, _x*_y + 1, _x)]
for f, g, vv in cas:
    assert expand(prem_s(f, g, vv) - prem(f, g, vv)) == 0
print(f"Cross-check OK : prem_s == sympy.prem sur {len(cas)} cas")

Cross-check OK : prem_s == sympy.prem sur 3 cas


In [5]:
def main_var(f, varlist):
    '''Variable principale : la plus grande variable (ordre d'elimination) qui apparait.'''
    for vv in reversed(varlist):
        if Poly(f, vv).degree() > 0:
            return vv
    return None

def rank_p(p_, varlist):
    '''(indice de la variable principale, degre en celle-ci) -- ordre lexicographique.'''
    mv = main_var(p_, varlist)
    return (varlist.index(mv), Poly(p_, mv).degree()) if mv else (-1, 0)

def basic_set(H, varlist):
    '''Selection gloutonne (Chou) : un polynome par variable principale, par rang croissant.'''
    B, used = [], set()
    for p_ in sorted(H, key=lambda z: rank_p(z, varlist)):
        mv = main_var(p_, varlist)
        if mv is None:
            return [p_]
        if mv not in used:
            B.append(p_)
            used.add(mv)
    return B

def char_set_basic(H, varlist, max_iter=80):
    '''Iteration de Chou : basic-set + pseudo-reduction des non-retenus jusqu'a stabilite.'''
    H = [expand(h) for h in H]
    for it in range(max_iter):
        B = basic_set(H, varlist)
        R = []
        for h in H:
            if h in B:
                continue
            rr = h
            for p_ in reversed(B):
                mv = main_var(p_, varlist)
                rr = prem_s(rr, p_, mv) if mv else rr
            if rr != 0:
                R.append(rr)
        if not R:
            return B, it
        H = sorted(set(H).union(R), key=lambda z: (rank_p(z, varlist), str(z)))
    raise RuntimeError("pas de convergence de l'ensemble caracteristique")

def wu_prem(g, B, varlist):
    '''Pseudo-reste successif de g a travers la chaine B, du sommet vers la base.'''
    R = expand(g)
    for p_ in reversed(B):
        mv = main_var(p_, varlist)
        R = prem_s(R, p_, mv) if mv else R
    return R

def initials(B, varlist):
    '''Initiales de la chaine : coefficients dominants en la variable principale.'''
    out = []
    for b in B:
        mv = main_var(b, varlist)
        if mv is not None:
            out.append(Poly(b, mv).LC())
    return out

print("noyau Wu pret : prem_s, main_var/rank_p, basic_set, char_set_basic, wu_prem, initials")

noyau Wu pret : prem_s, main_var/rank_p, basic_set, char_set_basic, wu_prem, initials


In [6]:
# La chaine caracteristique du systeme P1.
t0 = time.time()
B_IMO, it_IMO = char_set_basic(H_IMO, V_IMO)
print(f"CS(P1) : {len(B_IMO)} polynomes, converges en {it_IMO} iteration(s)   [{time.time()-t0:.3f}s]")
for b in B_IMO:
    print(f"   var principale {str(main_var(b, V_IMO)):2s} : {str(b)[:64]}")

CS(P1) : 5 polynomes, converges en 0 iteration(s)   [0.081s]


   var principale s  : s**2*v**2 - u**4 + 2*u**3 - u**2*v**2 - u**2 + u*v**2 - v**2/4
   var principale p  : 4*p**2*u**2*v**2 - 8*p**2*u*v**2 + 4*p**2*v**4 + 4*p**2*v**2 - 4
   var principale q  : 4*q**2*u**2*v**2 - 8*q**2*u*v**2 + 4*q**2*v**4 + 4*q**2*v**2 - 4
   var principale r  : 4*r**2*u**2*v**2 + 4*r**2*v**4 - 4*r*u**2*v**2 - 4*r*v**4 - 4*u*
   var principale t  : 4*t**2*u**2*v**2 + 4*t**2*v**4 - 4*t*u**2*v**2 - 4*t*v**4 - 4*u*


In [7]:
# La conclusion, famille par famille : chaque point B1, B2, C1, C2 est a la meme
# distance^2 du circoncentre O que A1 (qui y est par symetrie, comme A2).
w = w_sym
O = (Rational(1, 2), w)

def dist2_O(px, py):
    return (px - Rational(1, 2))**2 + (py - w)**2

ref = s**2 + w**2                       # d^2(A1, O) = d^2(A2, O) : symetrie autour de x=1/2
conclusions = {
    "B1": expand(together(ref - dist2_O(1 + p*(u - 1), p*v)).as_numer_denom()[0]),
    "B2": expand(together(ref - dist2_O(1 + q*(u - 1), q*v)).as_numer_denom()[0]),
    "C1": expand(together(ref - dist2_O(r*u, r*v)).as_numer_denom()[0]),
    "C2": expand(together(ref - dist2_O(t*u, t*v)).as_numer_denom()[0]),
}

t0 = time.time()
restes = {nom: wu_prem(g, B_IMO, V_IMO) for nom, g in conclusions.items()}
dt = time.time() - t0
for nom, R in restes.items():
    print(f"prem(g_{nom}, CS) = {R}   -> {'PROUVE' if R == 0 else 'NON NUL'}")
ok = all(R == 0 for R in restes.values())
print(f"\nVERDICT IMO 2008 P1 (enonce officiel, SIX points concycliques) : "
      f"{'PROUVE' if ok else 'NON CONCLUANT'}   [{dt:.3f}s]")
print("En bonus algebraique : le cercle commun est CENTRE AU CIRCONCENTRE de ABC.")

prem(g_B1, CS) = 0   -> PROUVE
prem(g_B2, CS) = 0   -> PROUVE
prem(g_C1, CS) = 0   -> PROUVE
prem(g_C2, CS) = 0   -> PROUVE

VERDICT IMO 2008 P1 (enonce officiel, SIX points concycliques) : PROUVE   [0.312s]
En bonus algebraique : le cercle commun est CENTRE AU CIRCONCENTRE de ABC.


In [8]:
# Les non-degenerescences produites par la preuve : les initiales de la chaine.
init_IMO = initials(B_IMO, V_IMO)
J_IMO = 1
for i in init_IMO:
    if not i.is_number:
        J_IMO = expand(J_IMO * i)
    print("   initiale :", factor(i))
print("\nLecture geometrique :")
print("   v^2        : v != 0        -> le triangle n'est pas aplati (A hors de (BC))")
print("   (u-1)^2+v^2 = |CA|^2       -> A != C : le trinome de la famille B est bien de degre 2")
print("   u^2+v^2    = |AB|^2        -> A != B : idem famille C")

   initiale : v**2
   initiale : 4*v**2*(u**2 - 2*u + v**2 + 1)
   initiale : 4*v**2*(u**2 - 2*u + v**2 + 1)
   initiale : 4*v**2*(u**2 + v**2)
   initiale : 4*v**2*(u**2 + v**2)

Lecture geometrique :
   v^2        : v != 0        -> le triangle n'est pas aplati (A hors de (BC))
   (u-1)^2+v^2 = |CA|^2       -> A != C : le trinome de la famille B est bien de degre 2
   u^2+v^2    = |AB|^2        -> A != B : idem famille C


In [9]:
def groebner_sat_verify(H, g, J, gens):
    '''g s'annule sur V(H) hors V(J) ssi GB([H, z*J-1, w*g-1]) = [1] (Rabinowitsch).'''
    z, wloc = symbols("_z _w")
    eqs = list(H) + [z*J - 1, wloc*g - 1]
    G = groebner(eqs, *(list(gens) + [z, wloc]), order="lex")
    return len(G.polys) == 1 and G.polys[0].as_expr() == 1

# Deux moteurs independants doivent conclure pareil (la regle des trois portes du 03).
t0 = time.time()
gb_B = groebner_sat_verify([h1, fB_p], conclusions["B1"], 1, [u, v, s, p])
dt_B = time.time() - t0
t0 = time.time()
gb_C = groebner_sat_verify([h1, fC_r], conclusions["C1"], 1, [u, v, s, r])
dt_C = time.time() - t0
print(f"Groebner-Rabinowitsch, famille B : g_B1 dans le radical -> {gb_B}   [{dt_B:.2f}s]")
print(f"Groebner-Rabinowitsch, famille C : g_C1 dans le radical -> {gb_C}   [{dt_C:.2f}s]")
accord = (restes['B1'] == 0) == gb_B and (restes['C1'] == 0) == gb_C
print("Deux moteurs independants (Wu, Groebner) concluent :", "ACCORD" if accord else "DESACCORD")

Groebner-Rabinowitsch, famille B : g_B1 dans le radical -> True   [0.01s]
Groebner-Rabinowitsch, famille C : g_C1 dans le radical -> True   [0.01s]
Deux moteurs independants (Wu, Groebner) concluent : ACCORD


### Lecture du résultat

**Ce que la preuve établit.** Les quatre pseudo-restes sont nuls : B₁, B₂, C₁ et C₂ sont chacun à la distance² s² + w² du point (1/2, w). A₁ et A₂ y sont par symétrie. Les six points de l'énoncé officiel sont donc concycliques — et l'algèbre offre plus que la question : le centre du cercle des six points est **le circoncentre de ABC**. (Une identité vectorielle le confirme a posteriori : pour chacun des six points X, |X − O|² = |H − O|² − R² − S où R est le rayon circonscrit et S = A·B + B·C + C·A — le sanity check numérique du §3 l'a mesurée à zéro exact.)

**Ce que Wu garantit.** La preuve vaut modulo les initiales non nulles — ici v ≠ 0, |CA| ≠ 0, |AB| ≠ 0 : triangle non aplati, sommets distincts. Ce sont exactement les cas où l'énoncé lui-même cesse de parler de quelque chose (« l'orthocentre », « le cercle de centre le milieu »). La version Gröbner-Rabinowitsch, elle, est passée **sans aucune condition** (J = 1) : sur cet encodage la conclusion est même inconditionnelle.

**Le coût, honnêtement.** Fractions de seconde ici, contre la boîte « 5 minutes par problème » du papier. L'écart n'est pas une victoire de notre `char_set_basic` sur le JGEX des auteurs : il mesure la différence entre **un encodage guidé** (nous avons choisi le repère, éliminé les racines par contraintes, visé le circoncentre) et **leur traduction générale et automatique** de l'énoncé entier en JGEX. La leçon reste : sur un problème bien posé, l'élimination de Wu est d'un bon rapport coût/preuve — c'est le constat de Sinha et al., reproduit ici à l'échelle d'un carnet.

## 5. Le littéral d'AG-30 : le déterminant `CYCLIC` à quatre points

Le benchmark n'évalue pas « six points concycliques » mais, pour P1B, `CYCLIC C1 C2 B1 A1`. La traduction algébrique canonique d'une cyclicité à quatre points P₁…P₄ est le **déterminant** de la matrice dont chaque ligne est (x² + y², x, y, 1) : s'annuler aux quatre points équivaut à être sur un même cercle (ou une droite — la non-dégénérescence usuelle). Ce polynôme est plus gros que nos différences de distances au circoncentre : il ne sait pas, a priori, où est le centre. C'est **la conclusion telle qu'un évaluateur général la poserait** — et la chaîne du §4 doit savoir la réduire aussi.

In [10]:
def ligne_cercle(px, py):
    return [expand(px**2 + py**2), expand(px), expand(py), 1]

def det_cyclique(P1, P2, P3, P4):
    return expand(Matrix([ligne_cercle(*P1), ligne_cercle(*P2),
                          ligne_cercle(*P3), ligne_cercle(*P4)]).det())

A1_pt = (Rational(1, 2) + s, 0)
B1_pt = (1 + p*(u - 1), p*v)
C1_pt, C2_pt = (r*u, r*v), (t*u, t*v)

gP1B = det_cyclique(C1_pt, C2_pt, B1_pt, A1_pt)          # SHOW: CYCLIC C1 C2 B1 A1
print("det CYCLIC C1 C2 B1 A1 :", len(str(gP1B)), "caracteres, degre total",
      Poly(gP1B, *V_IMO).total_degree())

# Sanity numerique du determinant : sur un triangle concret il s'annule aux six points.
_u_, _v_ = Rational(3, 7), Rational(5, 7)
pts_concrets = {nom: (x_, y_) for nom, x_, y_ in six_points(_u_, _v_)}
sub = {u: _u_, v: _v_, r: (pts_concrets['C1'][0]/_u_), t: (pts_concrets['C2'][0]/_u_),
       p: ((pts_concrets['B1'][1])/_v_), s: (pts_concrets['A1'][0] - Rational(1, 2))}
print("det sur figure concrete =", nsimplify(gP1B.subs(sub)))

t0 = time.time()
R_P1B = wu_prem(gP1B, B_IMO, V_IMO)
dt = time.time() - t0
print(f"\nprem(det_P1B, CS) = {R_P1B}   [{dt:.2f}s]")
print("VERDICT P1B litteral (CYCLIC C1 C2 B1 A1) :",
      "PROUVE" if R_P1B == 0 else "NON NUL")

det CYCLIC C1 C2 B1 A1 : 741 caracteres, degre total 7


det sur figure concrete = 0



prem(det_P1B, CS) = 0   [2.39s]
VERDICT P1B litteral (CYCLIC C1 C2 B1 A1) : PROUVE


## 6. La frontière de DD : ce que le combinatoire ne peut même pas poser

Le carnet [04](Geometry-04-DD-AR-Python.ipynb) a construit la base de déduction : des faits, des règles, une fermeture. Sur le fil rouge (milieu de l'hypoténuse), DD démontrait tout le combinatoire et s'arrêtait devant le métrique — AR prenait le relais. Sur **cette** configuration, la frontière est plus profonde : les points A₁…C₂ sont des *intersections d'un cercle et d'une droite*, et le corpus de règles n'a **aucune règle qui pose de tels points**. Ce n'est pas une lenteur : c'est une absence dans le langage.

Le corpus minimal ci-dessous (repris du 04) connaît milieux, alignements, perpendicularités et égalités de longueurs — voyons ce qu'il dérive de la construction, et ce qu'il ne touchera jamais.

In [11]:
def canon(fact):
    """Forme canonique d'un fait : les symetries d'arguments ne creent pas de fait nouveau."""
    name, args = fact[0], fact[1:]
    if name == "cong":
        e1, e2 = tuple(sorted(args[:2])), tuple(sorted(args[2:]))
        lo, hi = sorted((e1, e2))
        return ("cong",) + lo + hi
    if name == "coll":
        return ("coll",) + tuple(sorted(args))
    if name in ("perp", "para"):
        l1, l2 = tuple(sorted(args[:2])), tuple(sorted(args[2:]))
        lo, hi = sorted((l1, l2))
        return (name,) + lo + hi
    if name == "midp":
        return ("midp", args[0]) + tuple(sorted(args[1:]))
    return fact

def pretty(fact):
    name, a = fact[0], fact[1:]
    if name == "cong":
        return f"|{a[0]}{a[1]}| = |{a[2]}{a[3]}|"
    if name == "coll":
        return f"{a[0]}, {a[1]}, {a[2]} alignes"
    if name == "perp":
        return f"({a[0]}{a[1]}) _|_ ({a[2]}{a[3]})"
    if name == "para":
        return f"({a[0]}{a[1]}) // ({a[2]}{a[3]})"
    if name == "midp":
        return f"{a[0]} milieu de [{a[1]}{a[2]}]"
    return str(fact)

def rule_midp_def(base):
    """midp(M,A,B) -> cong(A,M,M,B)."""
    out = {}
    for f in base:
        if f[0] == "midp":
            m, a, b = f[1], f[2], f[3]
            out[canon(("cong", a, m, m, b))] = ("midp_def", (f,))
    return out

def rule_midp_coll(base):
    """midp(M,A,B) -> coll(A,M,B)."""
    out = {}
    for f in base:
        if f[0] == "midp":
            m, a, b = f[1], f[2], f[3]
            out[canon(("coll", a, m, b))] = ("midp_coll", (f,))
    return out

def rule_perp_para(base):
    """perp(l1,l2) et perp(l1,l3) -> para(l2,l3)."""
    out = {}
    perps = [f for f in base if f[0] == "perp"]
    for f1 in perps:
        l1, l2 = (f1[1], f1[2]), (f1[3], f1[4])
        for f2 in perps:
            l3, l4 = (f2[1], f2[2]), (f2[3], f2[4])
            if l1 == l3 and l2 != l4:
                out[canon(("para",) + l2 + l4)] = ("perp_para", (f1, f2))
            if l1 == l4 and l2 != l3:
                out[canon(("para",) + l2 + l3)] = ("perp_para", (f1, f2))
    return out

def rule_cong_trans(base):
    """cong(A,B,C,D) et cong(C,D,E,F) -> cong(A,B,E,F)."""
    out = {}
    by_edge = {}
    for f in base:
        if f[0] == "cong":
            e1, e2 = (f[1], f[2]), (f[3], f[4])
            by_edge.setdefault(e1, set()).add((f, e2))
            by_edge.setdefault(e2, set()).add((f, e1))
    for e, voisins in by_edge.items():
        for f1, e2 in voisins:
            for f2, e3 in by_edge.get(e2, ()):
                if e3 != e:
                    out[canon(("cong",) + e + e3)] = ("cong_trans", (f1, f2))
    return out

def closure(facts, rules, max_rounds=20):
    """Forward chaining jusqu'au point fixe (du carnet 04)."""
    base = {canon(f): ("hypothese", ()) for f in facts}
    rounds = 0
    changed = True
    while changed and rounds < max_rounds:
        changed = False
        rounds += 1
        for rule in rules:
            for fact, (rname, parents) in rule(base).items():
                if fact not in base:
                    base[fact] = (rule.__name__.replace("rule_", ""), parents)
                    changed = True
    return base, rounds

RULES = [rule_midp_def, rule_midp_coll, rule_perp_para, rule_cong_trans]

# Les faits de la construction P1, dans le langage de DD :
faits_P1 = [
    ("midp", "Ma", "B", "C"), ("midp", "Mb", "C", "A"), ("midp", "Mc", "A", "B"),
    ("perp", "A", "H", "B", "C"),      # l'altitude issue de A passe par H
    ("perp", "B", "H", "C", "A"),
    ("perp", "C", "H", "A", "B"),
]
base, rounds = closure(faits_P1, RULES)
derive = [f for f, (rn, _) in base.items() if rn != "hypothese"]
print(f"fermeture en {rounds} tours : {len(base)} faits, dont {len(derive)} derives")
for f in sorted(derive)[:8]:
    print(f"   [{base[f][0]:10s}] {pretty(f)}")
if len(derive) > 8:
    print(f"   ... et {len(derive)-8} autres")

mentions = [f for f in base if any(pt in f for pt in ("A1", "A2", "B1", "B2", "C1", "C2"))]
print(f"\nFaits de la fermeture mentionnant A1..C2 : {len(mentions)}")
print("DD ne peut pas les POSER : aucune regle du corpus ne cree l'intersection d'un")
print("cercle et d'une droite. La frontiere n'est pas une lenteur, c'est le langage --")
print("et c'est exactement le vide que le proposeur neuronal d'AlphaGeometry vient remplir.")

fermeture en 2 tours : 12 faits, dont 6 derives
   [midp_coll ] A, B, Mc alignes
   [midp_coll ] A, C, Mb alignes
   [midp_coll ] B, C, Ma alignes
   [midp_def  ] |AMb| = |CMb|
   [midp_def  ] |AMc| = |BMc|
   [midp_def  ] |BMa| = |CMa|

Faits de la fermeture mentionnant A1..C2 : 0
DD ne peut pas les POSER : aucune regle du corpus ne cree l'intersection d'un
cercle et d'une droite. La frontiere n'est pas une lenteur, c'est le langage --
et c'est exactement le vide que le proposeur neuronal d'AlphaGeometry vient remplir.


## 7. Témoin négatif : la même construction, un énoncé faux

La règle de la série : un moteur de preuve qui n'échoue jamais ne prouve rien. Variante fallacieuse : Γ_A passe **par A** au lieu de passer par H (les familles B et C restent définies par H). L'énoncé « les six points sont concycliques » devient **faux en général** — et les trois instruments du carnet doivent le dire chacun à sa manière : un contre-exemple numérique exact, un pseudo-reste non nul (Wu refuse), une non-appartenance au radical (Gröbner confirme le refus).

In [12]:
# Variante FAUSSE : Gamma_A passee par A (rayon |A - M_a|), familles B/C inchangees.
s2p = expand((u - Rational(1, 2))**2 + v**2)      # rayon^2 de Gamma_A' = |A - M_a|^2
h1f = expand(s**2 - s2p)                          # A1'=(1/2+s,0), A2'=(1/2-s,0)

gF = conclusions["B1"]                            # meme conclusion : B1 sur le cercle centre O

# 1. Contre-exemple numerique exact : sur un triangle concret, le cercle ne passe pas.
_u_, _v_ = Rational(3, 7), Rational(5, 7)
sa_f = sqrt(s2p.subs({u: _u_, v: _v_}))
w_ = w_sym.subs({u: _u_, v: _v_})
racines_F = solve(Poly(trinome_B(symbols("z")).subs({u: _u_, v: _v_}), symbols("z")), symbols("z"))
B1_F = (1 + racines_F[0]*(_u_ - 1), racines_F[0]*_v_)
dA1 = sa_f**2 + w_**2
dB1 = (B1_F[0] - Rational(1, 2))**2 + (B1_F[1] - w_)**2
print(f"1. triangle A=(3/7,5/7) : d^2(A1',O) = {dA1}  vs  d^2(B1,O) = {nsimplify(dB1)}")
print(f"   difference exacte NON NULLE : {nsimplify(dA1 - dB1)}  -> l'enonce est FAUX")
diff_num = float(dA1 - dB1)
assert diff_num != 0                        # ecart reel, pas une expression non simplifiee
print(f"   valeur numerique : {diff_num:.4f}")

# 2. Wu refuse : la chaine de la variante ne reduit pas la conclusion a zero.
B_F, _ = char_set_basic([h1f, fB_p, fB_q, fC_r, fC_t], V_IMO)
R_F = wu_prem(gF, B_F, V_IMO)
print(f"\n2. prem(g_B1, CS(variante)) = {R_F}")
print("   NON NUL -> Wu ne prouve pas l'enonce faux (refus, pas faux positif)")

# 3. Groebner confirme : la conclusion n'est pas dans le radical.
gb_F = groebner_sat_verify([h1f, fB_p], gF, 1, [u, v, s, p])
print(f"\n3. Groebner-Rabinowitsch : g dans le radical -> {gb_F} (False = non-derivabilite)")
print("\nTriple rejet coherent : numerique exact, pseudo-reste non nul, non-appartenance.")

1. triangle A=(3/7,5/7) : d^2(A1',O) = 1347/2450  vs  d^2(B1,O) = 11/70
   difference exacte NON NULLE : -(2*sqrt(16441)/1435 + 3/14)**2 - (6/35 - sqrt(16441)/574)**2 + 1347/2450  -> l'enonce est FAUX
   valeur numerique : 0.3927



2. prem(g_B1, CS(variante)) = -16*u**6*v**2 + 64*u**5*v**2 - 16*u**4*v**4 - 96*u**4*v**2 + 32*u**3*v**4 + 64*u**3*v**2 + 16*u**2*v**6 - 16*u**2*v**4 - 16*u**2*v**2 - 32*u*v**6 + 16*v**8 + 16*v**6
   NON NUL -> Wu ne prouve pas l'enonce faux (refus, pas faux positif)



3. Groebner-Rabinowitsch : g dans le radical -> False (False = non-derivabilite)

Triple rejet coherent : numerique exact, pseudo-reste non nul, non-appartenance.


## 8. Le proposeur neuronal — à échelle de jouet, déclarée comme telle

La brique qui sépare DD+AR de AlphaGeometry n'est pas un meilleur moteur de preuve : c'est le **proposeur**. Le transformer d'AG, entraîné sur 100 millions de preuves synthétiques, suggère les constructions auxiliaires qui débloquent la fermeture DD — c'est lui qui « voit » qu'il faut poser le circoncentre. Son entraînement est hors budget d'un carnet (GPU, jours de calcul, corpus de cent millions d'exemples).

Ce que nous pouvons démontrer honnêtement, c'est le **mécanisme** à échelle de jouet : un petit modèle **linéaire** qui, sans résoudre, prédit une quantity géométrique utile depuis la forme du triangle — et dont la prédiction est **vérifiée ensuite** par les moteurs exacts du carnet. La tâche jouet : parmi les trois familles {A₁A₂, B₁B₂, C₁C₂}, prédire celle dont les deux points sont **les plus proches** (la corde la plus courte) — une quantité qui dépend réellement de la forme du triangle, calculable exactement par les trinômes du §3, et qui joue ici le rôle de « quelle construction regarder en premier ». Entraînement sur des configurations générées par les moteurs du carnet, quelques secondes de CPU, `numpy` seul.

**Échelle, honnêtement** : un classifieur linéaire à 7 traits sur 2 000 exemples est à environ **10⁻⁷** de l'échelle d'entraînement d'AG (10⁸ exemples, transformer à centaines de millions de paramètres). Il ne démontre aucune compétence géométrique d'AG — il démontre la boucle *proposer puis vérifier*, et pourquoi un prior appris peut orienter une recherche exacte.

In [13]:
# Generateur exact : cordes des trois familles par forme fermee (coefficients des trinomes).
a2B, a1B, a0B = [lambdify((u, v), c, "numpy") for c in Poly(fB_p, p).all_coeffs()]
a2C, a1C, a0C = [lambdify((u, v), c, "numpy") for c in Poly(fC_r, r).all_coeffs()]
sa2 = lambdify((u, v), (u - Rational(1, 2))**2 + (u*(1 - u)/v)**2, "numpy")

def cordes(U, V):
    """Longueurs des cordes A1A2, B1B2, C1C2 pour des triangles (U, V) vectorises."""
    ca = 2*np.sqrt(sa2(U, V))
    discB = a1B(U, V)**2 - 4*a2B(U, V)*a0B(U, V)
    ca_b = np.sqrt(np.maximum(discB, 0))/a2B(U, V)*np.sqrt((U - 1)**2 + V**2)
    discC = a1C(U, V)**2 - 4*a2C(U, V)*a0C(U, V)
    ca_c = np.sqrt(np.maximum(discC, 0))/a2C(U, V)*np.sqrt(U**2 + V**2)
    return np.stack([ca, ca_b, ca_c], axis=1)

# Controle du generateur contre la resolution exacte (honetete de la fabrique) :
rng = np.random.default_rng(42)
ecarts_ctrl = []
for _ in range(3):
    u_ = float(rng.uniform(0.25, 0.75)); v_ = float(rng.uniform(0.6, 1.4))
    ru, rv = Rational(u_).limit_denominator(100), Rational(v_).limit_denominator(100)
    zz = symbols("z")
    racines_exact = solve(Poly(trinome_B(zz).subs({u: ru, v: rv}), zz), zz)
    diff_exacte = abs(racines_exact[0] - racines_exact[1])
    approx = cordes(np.array([u_]), np.array([v_]))[0, 1]/np.sqrt((u_ - 1)**2 + v_**2)
    ecarts_ctrl.append(abs(float(diff_exacte) - float(approx)))
print(f"controle generateur : forme fermee vs resolution exacte, ecart max = {max(ecarts_ctrl):.2e}")

# Donnees : 2000 triangles acutangles, etiquette = famille a la corde la plus courte.
N = 2000
U = rng.uniform(0.15, 0.85, N)
V = rng.uniform(0.55, 1.5, N)
acut = V**2 > U*(1 - U)                         # angle en A aigu
U, V = U[acut], V[acut]
C = cordes(U, V)
y = np.argmin(C, axis=1)

# Traits : la forme du triangle, SANS les cordes (qui sont la reponse).
cosB = U/np.sqrt(U**2 + V**2)
cosC = (1 - U)/np.sqrt((1 - U)**2 + V**2)
X = np.stack([U, V, U*(1 - U)/V, U**2 + V**2, (1 - U)**2 + V**2, cosB, cosC], axis=1)

n_tr = int(0.8*len(y))
Xtr, ytr, Xte, yte = X[:n_tr], y[:n_tr], X[n_tr:], y[n_tr:]
print(f"configurations : {len(y)} (acutangles), repartition des classes : "
      f"{np.bincount(y) / len(y)}")

# Proposeur jouet : regression softmax (numpy seul, descente de gradient).
Wm = np.zeros((Xtr.shape[1], 3)); bm = np.zeros(3)
mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-12
Xn, Xtn = (Xtr - mu)/sd, (Xte - mu)/sd
for _ in range(400):
    Z = Xn @ Wm + bm
    P = np.exp(Z - Z.max(1, keepdims=True)); P /= P.sum(1, keepdims=True)
    G = (P - np.eye(3)[ytr])/len(ytr)
    Wm -= 0.5*(Xn.T @ G); bm -= 0.5*G.sum(0)
acc_tr = (np.argmax(Xn @ Wm + bm, 1) == ytr).mean()
acc_te = (np.argmax(Xtn @ Wm + bm, 1) == yte).mean()
print(f"proposeur jouet : precis. train {acc_tr:.1%}, test {acc_te:.1%}")
print(f"baseline aleatoire uniforme : {1/3:.1%} -- le prior appris oriente la recherche")
print(f"baseline classeur majoritaire : {np.bincount(y).max()/len(y):.1%}")

controle generateur : forme fermee vs resolution exacte, ecart max = 1.01e-04
configurations : 2000 (acutangles), repartition des classes : [0.5885 0.2005 0.211 ]
proposeur jouet : precis. train 99.2%, test 99.2%
baseline aleatoire uniforme : 33.3% -- le prior appris oriente la recherche
baseline classeur majoritaire : 58.9%


## 9. Exercices

Les trois exercices étendent le carnet sur ses propres moteurs. Ils sont à compléter ; les stubs s'exécutent sans erreur (convention de la série : aucun carnet ne doit échouer sur un exercice non complété).

> **Exercice 1 — Le cercle des neuf points.** Encoder et prouver par le même noyau : *les trois milieux des côtés et les trois pieds des hauteurs de ABC sont concycliques.* Le pied de la hauteur issue de A est (u, 0) ; les milieux sont déjà dans le carnet. Quelle est la famille de cercles candidats cette fois ?

> **Exercice 2 — Mesurer la frontière de DD.** Ajouter au corpus DD une règle `rule_midp_cong_trans` exploitant les trois milieux (les trois « moitiés » de côtés), relancer la fermeture du §6 et mesurer : combien de faits dérivés, et toujours zéro mention de A₁…C₂ ? Comparer au temps Wu du §4 — c'est la mesure, pas l'impression, qui répond.

> **Exercice 3 — Étendre le proposeur.** Ajouter au classifieur du §8 deux traits (le produit cos B̂ · cos Ĉ, et v/√(u²+v²)) et une quatrième classe « la corde la plus longue ». Régénérer, réentraîner, comparer la précision test avant/après. Le gain mesure-t-il ce que coûte un trait inutile ?

In [14]:
# Exercice 1 -- le cercle des neuf points : a completer
# Etape 1 : milieux M_a=(1/2,0), M_b, M_c et pieds des hauteurs (pied de A : (u,0))
# Etape 2 : choix du cercle candidat (centre ? rayon ?) puis conclusions famille par famille
# Etape 3 : char_set_basic + wu_prem, puis groebner_sat_verify en croise
# TODO etudiant
resultat_ex1 = None    # a remplacer par le verdict du test de Wu
print("Exercice a completer")

Exercice a completer


In [15]:
# Exercice 2 -- mesurer la frontiere de DD : a completer
# Etape 1 : ecrire rule_midp_cong_trans (les moities de segments egaux se transmettent)
# Etape 2 : relancer closure(faits_P1, RULES + [nouvelle_regle]) et compter les faits derives
# Etape 3 : mesurer le temps du test de Wu au 4 et comparer les DEUX instruments
# TODO etudiant
resultat_ex2 = None    # a remplacer par (nombre de faits DD, temps Wu en secondes)
print("Exercice a completer")

Exercice a completer


In [16]:
# Exercice 3 -- etendre le proposeur jouet : a completer
# Etape 1 : ajouter les deux traits et la classe "corde la plus longue" (argmax au lieu d'argmin)
# Etape 2 : regenerer les donnees avec la meme graine, reentrainer la regression softmax
# Etape 3 : comparer la precis. test a 4 classes avec celle du carnet a 3 classes
# TODO etudiant
resultat_ex3 = None    # a remplacer par la precision test obtenue
print("Exercice a completer")

Exercice a completer


## Conclusion

Ce carnet a pris un problème qu'aucun système neuro-symbolique du benchmark ne résout — IMO 2008 P1B — et l'a traité avec les moteurs que la série a construits :

| Étape | Résultat | Où |
|---|---|---|
| Encodage | 5 hypothèses triangulaires, 2 paramètres — aucune racine carrée | §3 |
| Preuve Wu | **l'énoncé officiel à six points, prouvé** — en prime, le centre est le circoncentre | §4 |
| Non-dégénérescences | v ≠ 0, A ≠ B, A ≠ C — celles mêmes où l'énoncé se tait | §4 |
| Certificat croisé | Gröbner-Rabinowitsch : **ACCORD**, sans condition | §4 |
| Littéral du benchmark | déterminant `CYCLIC C1 C2 B1 A1` réduit à zéro par la même chaîne | §5 |
| Frontière DD | 0 fait sur A₁…C₂ : l'intersection cercle-droite n'est pas dans le langage | §6 |
| Témoin négatif | énoncé faux rejeté trois fois (numérique, Wu, Gröbner) | §7 |
| Proposeur jouet | la boucle proposer-vérifier démontrée, à 10⁻⁷ de l'échelle d'AG | §8 |

**La morale du benchmark.** La lecture superficielle de 2024 était « le neuronal a gagné ». La mesure de Sinha et al., repliée ici sur un seul problème, dit plus précisément : *le symbolique classique n'avait pas perdu*. Une méthode de 1977, sans aucun apprentissage, prouve en fractions de seconde (sur un encodage guidé) un énoncé qu'un transformer de cent millions d'exemples ne résout pas — et l'union des deux fait mieux que chacun. La complémentarité n'est pas un slogan : DD tient le combinatoire, AR tient le métrique local, Wu tient l'algèbre entière quand on sait l'encoder, le proposeur appris oriente. C'est la thèse du carnet [04](Geometry-04-DD-AR-Python.ipynb), portée à l'échelle d'un vrai sujet d'olympiade.

***

*Notebook de la série Geometry — SymbolicAI. Fait partie du cycle complet du raisonnement vérifiable du dépôt CoursIA.*

## À retenir

- **L'encodage tient** : 5 hypothèses triangulaires, 2 paramètres, aucune racine carrée, et le pire écart exact sur trois triangles est **0** — l'encodage est cohérent avec l'énoncé avant toute preuve.
- **L'énoncé officiel est prouvé** : les quatre restes `prem(g_B1, CS) = 0`, `prem(g_B2, CS) = 0`, `prem(g_C1, CS) = 0`, `prem(g_C2, CS) = 0` donnent `VERDICT IMO 2008 P1 (six points concycliques) : PROUVE` en 0.312 s — avec en prime le centre au circoncentre de `ABC`.
- **Les non-dégénérescences sont les positions où l'énoncé se tait** : `v^2` (triangle non aplati), `4*v^2*(u^2 + v^2)` (`A ≠ C`), `4*v^2*(u^2 - 2*u + v^2 + 1)` (`A ≠ B`).
- **Deux moteurs indépendants s'accordent** : Gröbner-Rabinowitsch rend `True` sur les deux familles en 0.01 s — `Deux moteurs independants (Wu, Groebner) concluent : ACCORD`.
- **Le littéral du benchmark est prouvé par la même chaîne** : le déterminant `CYCLIC C1 C2 B1 A1` (741 caractères, degré total 7) se réduit à zéro — `prem(det_P1B, CS) = 0` en 2.39 s.
- **La frontière de DD est mesurée** : la fermeture atteint 12 faits (dont 6 dérivés) mais **0** mentionne `A1…C2` — l'intersection cercle-droite n'est pas dans le langage des règles.
- **Le proposeur orienté la recherche** : précision test 99.2 % contre 33.3 % pour une baseline uniforme et 58.9 % pour le classifieur majoritaire, sur 2000 configurations acutangles.


## Vérifiez votre compréhension

1. Le pire écart exact sur trois triangles est `0`. Pourquoi cette mesure précède-t-elle la preuve, et que mesurerait-elle de faux ?
2. Les initiales de la chaîne sont `v^2`, `4*v^2*(u^2 + v^2)` et `4*v^2*(u^2 - 2*u + v^2 + 1)`. Que chacune dit-elle sur la géométrie de la configuration ?
3. Le même énoncé est prouvé par Wu **et** par Gröbner-Rabinowitsch. Que gagne-t-on, en termes de risque, à faire confirmer un résultat par un second moteur ?
4. La fermeture DD produit 12 faits dont aucun ne mentionne `A1…C2`. Comment lit-on cette absence : comme un échec du moteur, ou comme une **frontière de langage** ?

<details>
<summary>Réponses</summary>

1. Parce que la preuve est **inconditionnelle** : un encodage faux peut produire un reste nul — on prouve alors autre chose. Le sanity numérique vérifie que l'encodage et l'énoncé disent la même chose ; il ne prouve rien, il **écarte** le contresens. Un écart non nul dirait ici que l'encodage est fautif.
2. `v^2` dit que `v ≠ 0`, donc que le triangle n'est pas aplati (`A` hors de `(BC)`) ; `4*v^2*(u^2 + v^2)` que `A ≠ C` — le trinôme de la famille B est bien de degré 2 ; `4*v^2*(u^2 - 2*u + v^2 + 1)` que `A ≠ B`, par la même lecture via `|CA|^2 = (u-1)^2 + v^2`. Ce sont les positions où l'énoncé n'a pas de sens.
3. La **dépendance** : deux moteurs qui partagent une implémentation ne sont pas deux témoins. Wu et Gröbner-Rabinowitsch n'ont en commun que le système d'hypothèses ; leurs algorithmes diffèrent — élimination incrémentale contre saturation par Rabinowitsch. L'accord porte donc sur l'énoncé, pas sur une implémentation.
4. Comme une **frontière de langage** : les règles du corpus parlent de milieux, d'alignements et de congruences, et l'intersection cercle-droite n'a pas de correspondant parmi elles. Ajouter une règle pour la rendre accessible est possible, mais ce serait un autre langage — et le carnet 04 mesure déjà que DD tient le combinatoire tandis qu'AR tient le métrique.

</details>


## Pour aller plus loin

- **Sinha et al. 2024** (arXiv:2404.06405, PDF archivé dans le gisement commun) — la réévaluation complète : 15 (Wu seul) / 21 (Wu+DD+AR, argent) / 27 (Wu+AG, or), et les deux problèmes Wu-seuls.
- **Les artefacts du benchmark** — dépôt HuggingFace `bethgelab/simplegeometry` : les fichiers `.gex` JGEX de chaque problème, dont les deux entrées 2008 P1a/P1B décodées au §2.
- Dans la série : [Geometry-03b](Geometry-03b-Ritt-Decomposition-Python.ipynb) pousse le noyau Wu jusqu'à la décomposition de Ritt quand la variété est réductible ; [Geometry-05](Geometry-05-Pont-Formel-Python.ipynb) fait traverser au fil rouge la frontière du formel vérifié.

**Série Geometry** : [Geometry-05 — Le pont formel →](Geometry-05-Pont-Formel-Python.ipynb)
